# 02 — MVP Run (50 ids × 12 variations = 600 images)

**Hardware:** 4× L4 GPUs via Ray (cluster `6106-192556-lj0uddmy`)

**Config:** `configs/mvp.yaml` — NF4 + Flux + ControlNet (pose) + PuLID + antelopev2 + 1024², all validated against the smoke (`01_databricks_ray_smoke.ipynb`).

**Expected wall:** ~75 min on warm Volume cache (Flux ~24 GB already cached at `/Volumes/ds_work/alenj00/cap_cache/hf_cache/`).

**Idempotent:** if this run is interrupted, re-running picks up only the missing images. Per-image manifest streams to `incremental_manifest.jsonl` in the output dir.

## 1. Install cap (mediapipe pinned, PuLID via Volume)

In [ ]:
%pip install --force-reinstall --no-deps mediapipe==0.10.20
%pip install /Workspace/Users/alenj00@safeway.com/counterfactual-audit-pipeline

In [ ]:
dbutils.library.restartPython()

## 2. Volume cache + secrets

In [ ]:
import os
from pathlib import Path

HF_CACHE = "/Volumes/ds_work/alenj00/cap_cache/hf_cache"
PULID_SRC = "/Volumes/ds_work/alenj00/cap_cache/pulid_src"

os.environ["HF_HOME"] = HF_CACHE
os.environ["HUGGINGFACE_HUB_CACHE"] = HF_CACHE
os.environ["TRANSFORMERS_CACHE"] = HF_CACHE

HF_TOKEN = dbutils.secrets.get(scope="cap-secrets", key="hf-token")
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN

assert Path(HF_CACHE).exists(), f"HF cache missing: {HF_CACHE}"
assert Path(PULID_SRC, "pulid", "__init__.py").exists(), f"PuLID source missing: {PULID_SRC}"
print(f"HF cache:   {HF_CACHE} ✓")
print(f"PuLID src:  {PULID_SRC} ✓")
print(f"HF token:   set (length {len(HF_TOKEN)})")

## 3. Bring up Ray across all 4 L4s

In [ ]:
import ray
from ray.util.spark import setup_ray_cluster, shutdown_ray_cluster

try:
    shutdown_ray_cluster()
except Exception:
    pass

setup_ray_cluster(
    num_worker_nodes=3,
    num_gpus_worker_node=1,
    num_cpus_worker_node=8,
    num_gpus_head_node=1,
    num_cpus_head_node=4,
)
ray.init(ignore_reinit_error=True)
print("Ray cluster:", {k: v for k, v in sorted(ray.cluster_resources().items()) if k in ("GPU", "CPU", "memory")})

## 4. Run MVP via the cap CLI

Calls `cap.cli.generate.main` programmatically (so we can pass per-cluster paths via flags rather than baking them into the config). Idempotent — interrupted runs resume from missing images.

In [ ]:
import time
from click.testing import CliRunner
from cap.cli.generate import main as generate_cli

REPO = "/Workspace/Users/alenj00@safeway.com/counterfactual-audit-pipeline"

t0 = time.time()
runner = CliRunner()
result = runner.invoke(
    generate_cli,
    [
        "--config", f"{REPO}/configs/mvp.yaml",
        "--backend", "ray",
        "--num-actors", "4",
        "--cache-dir", HF_CACHE,
        "--pulid-src", PULID_SRC,
        "--hf-token", HF_TOKEN,
    ],
    catch_exceptions=False,
    standalone_mode=False,
)
wall = time.time() - t0
print(f"\nMVP wall: {wall:.0f}s ({wall/60:.1f} min)")
print("CLI output:")
print(result.output)

## 5. Verify output

In [ ]:
import pandas as pd
from pathlib import Path

GENERATED = Path("/Volumes/ds_work/alenj00/cap_cache/runs/mvp/generated")
pngs = sorted(GENERATED.glob("*.png"))
print(f"Generated {len(pngs)} PNG files in {GENERATED}")

manifest = GENERATED / "manifest.parquet"
if manifest.exists():
    df = pd.read_parquet(manifest)
    print(f"Manifest rows: {len(df)}")
    print("By axis combo:")
    if "axis_skin_tone" in df.columns and "axis_gender" in df.columns:
        print(df.groupby(["axis_skin_tone", "axis_gender"]).size().unstack(fill_value=0))
    print("\nFirst 3 rows:")
    print(df.head(3))

## 6. Cleanup

In [ ]:
shutdown_ray_cluster()
print("Ray cluster shut down. MVP complete.")